# Chapter 5 — RLHF Alignment Pipeline
## SFT → Reward Model → DPO → PPO (vanilla + mitigated)

This notebook imports all algorithms from the `rlbook` package.
For the zero-dependency verification notebook, see `ch5_verify_gpu_v2.ipynb`.


In [ ]:
# Colab setup (uncomment one):
# !pip install -q "git+https://github.com/rl-textbook/reinforcement-learning.git#egg=rlbook[alignment]"
# !pip install -q detoxify


In [ ]:
import copy, random, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, LogitsProcessor

from rlbook.algorithms.alignment import (
    Config, SFTTrainer, RewardModel, DPOTrainer, PPOTrainer,
    sequence_log_probs, response_log_probs, get_lr_lambda, filter_quality_pairs,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = Config()
print(f"Device: {device} | Model: {cfg.model_name}")


In [ ]:
# ── Tokenizer & data ──────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

dataset = load_dataset("Anthropic/hh-rlhf", split="train")

def extract_prompt(text):
    turns = text.split("\nAssistant:")
    return turns[0] + "\nAssistant:" if len(turns) > 1 else text[:200]

sft_data = list(dataset.select(range(cfg.sft_train_size + cfg.sft_val_size)))
pref_raw = dataset.select(range(cfg.sft_train_size + cfg.sft_val_size,
                                 cfg.sft_train_size + cfg.sft_val_size + cfg.pref_train_size + cfg.pref_val_size))
pref_list = filter_quality_pairs(
    [{"chosen": ex["chosen"], "rejected": ex["rejected"]} for ex in pref_raw],
    tokenizer, min_tokens=cfg.min_response_tokens)
print(f"SFT: {len(sft_data)} | Pref (filtered): {len(pref_list)}")


In [ ]:
# ── Collation ─────────────────────────────────────────────────────
def collate_sft(batch):
    encs = [tokenizer(ex["chosen"], truncation=True, max_length=cfg.max_length) for ex in batch]
    max_len = max(len(e["input_ids"]) for e in encs)
    def pad(seq, val): return [val]*(max_len-len(seq)) + seq  # left-pad
    ids = torch.tensor([pad(e["input_ids"], cfg.pad_token_id) for e in encs])
    mask = torch.tensor([pad(e["attention_mask"], 0) for e in encs])
    labels = ids.clone(); labels[mask == 0] = -100
    return {"input_ids": ids.to(device), "attention_mask": mask.to(device), "labels": labels.to(device)}

def tokenize_pair(ex):
    c = tokenizer(ex["chosen"], truncation=True, max_length=cfg.max_length, padding="max_length", return_tensors="pt")
    r = tokenizer(ex["rejected"], truncation=True, max_length=cfg.max_length, padding="max_length", return_tensors="pt")
    prompt = extract_prompt(ex["chosen"])
    pl = len(tokenizer(prompt, truncation=True, max_length=cfg.max_length)["input_ids"])
    return {"chosen_input_ids": c["input_ids"].squeeze(0), "chosen_attention_mask": c["attention_mask"].squeeze(0),
            "rejected_input_ids": r["input_ids"].squeeze(0), "rejected_attention_mask": r["attention_mask"].squeeze(0),
            "prompt_length": pl}

def collate_pref(batch):
    return {"chosen_input_ids": torch.stack([b["chosen_input_ids"] for b in batch]).to(device),
            "chosen_attention_mask": torch.stack([b["chosen_attention_mask"] for b in batch]).to(device),
            "rejected_input_ids": torch.stack([b["rejected_input_ids"] for b in batch]).to(device),
            "rejected_attention_mask": torch.stack([b["rejected_attention_mask"] for b in batch]).to(device),
            "prompt_lengths": torch.tensor([b["prompt_length"] for b in batch]).to(device)}

sft_loader = DataLoader(sft_data, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_sft)
pref_tok = [tokenize_pair(ex) for ex in pref_list]
pref_loader = DataLoader(pref_tok[:cfg.pref_train_size], batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_pref)
pref_val_loader = DataLoader(pref_tok[cfg.pref_train_size:], batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_pref)

def make_prompt_loader(data, bs):
    return DataLoader(data, batch_size=bs, shuffle=True,
                      collate_fn=lambda batch: tokenizer(
                          [extract_prompt(ex["chosen"]) for ex in batch],
                          return_tensors="pt", padding=True, truncation=True, max_length=128))
print("Data loaders ready.")


## Stage 1: SFT


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
base_model.resize_token_embeddings(len(tokenizer))
sft_trainer = SFTTrainer(base_model, lr=cfg.sft_lr)

print("=== SFT ===")
sft_it = iter(sft_loader)
for step in range(cfg.sft_steps):
    try: batch = next(sft_it)
    except StopIteration: sft_it = iter(sft_loader); batch = next(sft_it)
    loss = sft_trainer.train_step(batch)
    if (step+1) % 100 == 0: print(f"  Step {step+1}/{cfg.sft_steps} | loss={loss:.4f}")
sft_model = base_model
del sft_trainer; torch.cuda.empty_cache()
print("SFT done.")


## Stage 2: Reward Model


In [ ]:
rm_base = AutoModelForCausalLM.from_pretrained(cfg.model_name)
rm_base.resize_token_embeddings(len(tokenizer)); rm_base.to(device)
hidden_size = rm_base.transformer.wte.weight.size(1)
reward_model = RewardModel(rm_base, hidden_size).to(device)
rm_opt = AdamW(reward_model.parameters(), lr=cfg.rm_lr)
warmup = int(cfg.rm_steps * cfg.warmup_fraction)
rm_sched = LambdaLR(rm_opt, get_lr_lambda(warmup, cfg.rm_steps))

print("=== RM Training ===")
rm_it = iter(pref_loader)
for step in range(cfg.rm_steps):
    try: batch = next(rm_it)
    except StopIteration: rm_it = iter(pref_loader); batch = next(rm_it)
    reward_model.train()
    rc = reward_model(batch["chosen_input_ids"], batch["chosen_attention_mask"])
    rr = reward_model(batch["rejected_input_ids"], batch["rejected_attention_mask"])
    loss = -F.logsigmoid(rc - rr).mean()
    rm_opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
    rm_opt.step(); rm_sched.step()
    if (step+1) % 200 == 0:
        print(f"  Step {step+1}/{cfg.rm_steps} | loss={loss.item():.4f} | acc={(rc>rr).float().mean():.3f}")

reward_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for b in pref_val_loader:
        rc = reward_model(b["chosen_input_ids"], b["chosen_attention_mask"])
        rr = reward_model(b["rejected_input_ids"], b["rejected_attention_mask"])
        correct += (rc > rr).sum().item(); total += rc.size(0)
print(f"RM held-out accuracy: {correct/total:.3f}")
for p in reward_model.parameters(): p.requires_grad_(False)
del rm_opt; torch.cuda.empty_cache()


## Stage 3: DPO


In [ ]:
dpo_policy = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
dpo_policy.resize_token_embeddings(len(tokenizer))
dpo_policy.load_state_dict(sft_model.state_dict())
ref_policy = copy.deepcopy(dpo_policy).eval()
for p in ref_policy.parameters(): p.requires_grad_(False)
dpo_trainer = DPOTrainer(dpo_policy, ref_policy, beta=cfg.beta_dpo, lr=cfg.dpo_lr)

print("=== DPO ===")
dpo_it = iter(pref_loader)
for step in range(cfg.dpo_steps):
    try: batch = next(dpo_it)
    except StopIteration: dpo_it = iter(pref_loader); batch = next(dpo_it)
    loss, margin = dpo_trainer.train_step(batch)
    if (step+1) % 200 == 0: print(f"  Step {step+1}/{cfg.dpo_steps} | loss={loss:.4f} | margin={margin:.4f}")

dpo_state = {k: v.cpu() for k, v in dpo_policy.state_dict().items()}
del dpo_policy, ref_policy, dpo_trainer; torch.cuda.empty_cache()
print("DPO done.")


## Stage 4: Vanilla PPO


In [ ]:
ppo_policy = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
ppo_policy.resize_token_embeddings(len(tokenizer))
ppo_policy.load_state_dict(sft_model.state_dict())
ppo_ref = copy.deepcopy(ppo_policy).eval()
for p in ppo_ref.parameters(): p.requires_grad_(False)

ppo = PPOTrainer(ppo_policy, ppo_ref, reward_model, tokenizer, cfg,
                 kl_coef=cfg.kl_coef_ppo, repetition_penalty=1.0, reward_clip=None)
nn.init.zeros_(ppo.value_head.weight); nn.init.zeros_(ppo.value_head.bias)

# Warmup normalizer
for wb in DataLoader(sft_data[:40], batch_size=cfg.ppo_batch_size, shuffle=False,
                     collate_fn=lambda b: tokenizer([extract_prompt(e["chosen"]) for e in b],
                                                     return_tensors="pt", padding=True, truncation=True, max_length=128)):
    g, m, pl = ppo.generate_responses(wb); ppo.compute_rewards_and_old_logprobs(g, m, pl)

print(f"=== Vanilla PPO ({cfg.ppo_outer_steps} steps, kl_coef={cfg.kl_coef_ppo}) ===")
ppo_it = iter(make_prompt_loader(sft_data, cfg.ppo_batch_size))
nan_count = 0
for step in range(cfg.ppo_outer_steps):
    try: prompts = next(ppo_it)
    except StopIteration: ppo_it = iter(make_prompt_loader(sft_data, cfg.ppo_batch_size)); prompts = next(ppo_it)
    g, m, pl = ppo.generate_responses(prompts)
    r, olp = ppo.compute_rewards_and_old_logprobs(g, m, pl)
    loss, _, _ = ppo.ppo_update(g, m, pl, r, olp, num_epochs=cfg.ppo_inner_epochs)
    if np.isnan(loss):
        nan_count += 1
        if nan_count >= 20:
            print(f"[PPO] Stopping: {nan_count} NaN losses")
            break
        continue
    else:
        nan_count = max(0, nan_count - 1)
    if (step+1) % 100 == 0: print(f"  Step {step+1}/{cfg.ppo_outer_steps} | loss={loss:.4f}")
    if nan_count >= 20: break

vanilla_state = {k: v.cpu() for k, v in ppo_policy.state_dict().items()}
del ppo_policy, ppo_ref, ppo; torch.cuda.empty_cache()
print("Vanilla PPO done.")


## Stage 4b: Mitigated PPO
Higher KL (0.1), repetition penalty (1.2), reward clipping (±2.0).


In [ ]:
ppo_m = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
ppo_m.resize_token_embeddings(len(tokenizer)); ppo_m.load_state_dict(sft_model.state_dict())
ppo_m_ref = copy.deepcopy(ppo_m).eval()
for p in ppo_m_ref.parameters(): p.requires_grad_(False)

mit = PPOTrainer(ppo_m, ppo_m_ref, reward_model, tokenizer, cfg,
                 kl_coef=0.1, repetition_penalty=1.2, reward_clip=2.0)
nn.init.zeros_(mit.value_head.weight); nn.init.zeros_(mit.value_head.bias)
for wb in DataLoader(sft_data[:40], batch_size=cfg.ppo_batch_size, shuffle=False,
                     collate_fn=lambda b: tokenizer([extract_prompt(e["chosen"]) for e in b],
                                                     return_tensors="pt", padding=True, truncation=True, max_length=128)):
    g, m, pl = mit.generate_responses(wb); mit.compute_rewards_and_old_logprobs(g, m, pl)

print("=== Mitigated PPO (kl=0.1, rep=1.2, clip=±2.0) ===")
ppo_it = iter(make_prompt_loader(sft_data, cfg.ppo_batch_size))
nan_count = 0
for step in range(cfg.ppo_outer_steps):
    try: prompts = next(ppo_it)
    except StopIteration: ppo_it = iter(make_prompt_loader(sft_data, cfg.ppo_batch_size)); prompts = next(ppo_it)
    g, m, pl = mit.generate_responses(prompts)
    r, olp = mit.compute_rewards_and_old_logprobs(g, m, pl)
    loss, _, _ = mit.ppo_update(g, m, pl, r, olp, num_epochs=cfg.ppo_inner_epochs)
    if np.isnan(loss):
        nan_count += 1
        if nan_count >= 20:
            print(f"[MIT-PPO] Stopping: {nan_count} NaN losses")
            break
        continue
    else:
        nan_count = max(0, nan_count - 1)
    if (step+1) % 100 == 0: print(f"  Step {step+1}/{cfg.ppo_outer_steps} | loss={loss:.4f}")
    if nan_count >= 20: break

mit_state = {k: v.cpu() for k, v in ppo_m.state_dict().items()}
del ppo_m, ppo_m_ref, mit; torch.cuda.empty_cache()
print("Mitigated PPO done.")


## Stage 5: 4-Way Evaluation


In [ ]:
# Reload all models for eval
def load_eval(state):
    m = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
    m.resize_token_embeddings(len(tokenizer)); m.load_state_dict(state); m.eval()
    return m

models = {"SFT": sft_model.eval(), "DPO": load_eval(dpo_state),
          "PPO (vanilla)": load_eval(vanilla_state), "PPO (mitigated)": load_eval(mit_state)}

try:
    from detoxify import Detoxify; det = Detoxify("original", device=device)
except ImportError: det = None; print("detoxify not installed — skipping toxicity eval")

HARMFUL_KW = ["kill","harm","suicide","weapon","illegal","steal","attack","abuse","hate","threat","bomb","poison","drug"]
class _San(LogitsProcessor):
    def __call__(self, ids, s): return torch.where(torch.isfinite(s), s, torch.full_like(s, -1e9))

def gen(model, prompt):
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=64, do_sample=True, temperature=0.7,
                             pad_token_id=cfg.pad_token_id, logits_processor=[_San()])
    return tokenizer.decode(out[0][enc["input_ids"].size(1):], skip_special_tokens=True)

all_prompts = [extract_prompt(ex["chosen"]) for ex in sft_data]
harm_prompts = [p for p in all_prompts if any(k in p.lower() for k in HARMFUL_KW)][:50]
eval_prompts = all_prompts[:50]

print(f"Harmful: {len(harm_prompts)} | General: {len(eval_prompts)}")
for name, model in models.items():
    gh = [gen(model, p) for p in harm_prompts]
    ga = [gen(model, p) for p in eval_prompts]
    tox = det.predict(gh)["toxicity"] if det else [0.0]*len(gh)
    rm_h = np.mean([reward_model(*[t.to(device) for t in tokenizer(p+" "+r, return_tensors="pt",
                     truncation=True, max_length=cfg.max_length).values()]).item()
                     for p,r in zip(harm_prompts, gh)])
    rm_a = np.mean([reward_model(*[t.to(device) for t in tokenizer(p+" "+r, return_tensors="pt",
                     truncation=True, max_length=cfg.max_length).values()]).item()
                     for p,r in zip(eval_prompts, ga)])
    print(f"  {name:18s} | tox={np.mean(tox):.4f} safe={np.mean([t<0.5 for t in tox]):.3f} | rm={rm_a:.4f} rm_safety={rm_h:.4f}")
